# Week 7 - 03: RAG Testing and Grounding

A RAG project is not finished just because the code runs.

We test:
1. Questions covered by our documents.
2. Questions not covered by our documents.
3. Whether retrieved context is relevant.
4. The difference between an LLM answering with and without retrieved context.

This combines the useful testing ideas from the earlier Week 7 files without repeating several complete RAG implementations.


In [ ]:
!pip install chromadb sentence-transformers anthropic

In [1]:
import chromadb
from sentence_transformers import SentenceTransformer

from google import genai
from dotenv import load_dotenv
import os
load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Connect to the same database from notebooks 01 and 02.
db_client = chromadb.PersistentClient(path="./week7_rag_db")
collection = db_client.get_or_create_collection(name="documents")

print("Testing system is ready!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Testing system is ready!


In [3]:
def retrieve_context(question, top_k=3):
    # Converts the question into an embedding.
    question_embedding = embedding_model.encode(question).tolist()

    # Finds the closest chunks in ChromaDB.
    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=top_k
    )

    chunks = results["documents"][0]

    # Combines the chunks into the context given to Gemini.
    context = "\n".join(
        f"Context {i}: {chunk}"
        for i, chunk in enumerate(chunks, start=1)
    )

    return context


In [44]:
import time
from google.genai.errors import APIError

def ask_gemini(question, context=None):

    prompt = f"""
Answer the question using the retrieved context when it is relevant.

If the context does not contain the answer, use your general knowledge.

Retrieved context:
{context}

Question:
{question}

Give a concise and accurate answer.
"""

    models = [
        "gemini-3.6-flash",
        "gemini-3.7-flash"
    ]

    for model in models:
        for attempt in range(3):
            try:
                response = client.models.generate_content(
                    model=model,
                    contents=prompt
                )

                if response.text:
                    return response.text.strip()

                return "Gemini returned an empty answer."

            except APIError as error:
                error_text = str(error)

                if "503" in error_text or "UNAVAILABLE" in error_text:
                    wait_seconds = 2 ** attempt

                    print(
                        f"{model} is busy. "
                        f"Retrying in {wait_seconds} seconds..."
                    )

                    time.sleep(wait_seconds)
                else:
                    return f"Gemini API error: {error}"

    return "Both Gemini models are temporarily unavailable. Please try again later."

Test 1: Questions covered by our documents

In [35]:
questions = [
    "How can AI help with scheduling?",
    "What does a vector database store?",
    "What is semantic search?"
]

for question in questions:
    context = retrieve_context(question)

    print("=" * 70)
    print("QUESTION:", question)
    print("\nRETRIEVED CONTEXT:")
    print(context)
    print("\nANSWER:")
    print(ask_gemini(question, context))


QUESTION: How can AI help with scheduling?

RETRIEVED CONTEXT:
Context 1: AI scheduling can assign teachers, rooms, courses, and time slots while following constraints.
Context 2: Artificial Intelligence allows computers to perform tasks that normally require human intelligence.
Context 3: Machine learning allows a computer to learn patterns from examples and data.

ANSWER:
Based on the provided context, AI can help with scheduling by assigning teachers, rooms, courses, and time slots automatically while ensuring all specific constraints are followed.
QUESTION: What does a vector database store?

RETRIEVED CONTEXT:
Context 1: Vector databases store embeddings and support semantic similarity search.
Context 2: Machine learning allows a computer to learn patterns from examples and data.
Context 3: Semantic search retrieves information based on meaning rather than exact keywords.

ANSWER:
A vector database stores embeddings.
QUESTION: What is semantic search?

RETRIEVED CONTEXT:
Context 1

Test 2: Questions not covered by our documents

In [45]:
questions = [
    "Who is the current President of Nepal?",
    "What is the capital of France?"
]

for question in questions:
    context = retrieve_context(question)

    print("=" * 70)
    print("QUESTION:", question)
    print("\nRETRIEVED CONTEXT:")
    print(context)
    print("\nANSWER:")
    print(ask_gemini(question, context))


QUESTION: Who is the current President of Nepal?

RETRIEVED CONTEXT:
Context 1: Artificial Intelligence allows computers to perform tasks that normally require human intelligence.
Context 2: Natural language processing helps computers understand and generate human language.
Context 3: AI scheduling can assign teachers, rooms, courses, and time slots while following constraints.

ANSWER:
The current President of Nepal is Ram Chandra Poudel.
QUESTION: What is the capital of France?

RETRIEVED CONTEXT:
Context 1: AI scheduling can assign teachers, rooms, courses, and time slots while following constraints.
Context 2: Artificial Intelligence allows computers to perform tasks that normally require human intelligence.
Context 3: Semantic search retrieves information based on meaning rather than exact keywords.

ANSWER:
The capital of France is Paris.


Test 3: With context vs without context

In [46]:
question = "What is semantic search?"

context = retrieve_context(question)

print("=" * 70)
print("QUESTION:", question)

print("\nWITHOUT RETRIEVED CONTEXT:")
print(ask_gemini(question))

print("\nWITH RETRIEVED CONTEXT:")
print(ask_gemini(question, context))


QUESTION: What is semantic search?

WITHOUT RETRIEVED CONTEXT:
**Semantic search** is a search technique that seeks to understand the intent and contextual meaning behind a user's query, rather than relying solely on exact keyword matching. 

By using natural language processing (NLP) and vector embeddings, semantic search retrieves results based on the underlying concepts, relationships, and context of the words, delivering more relevant and accurate answers.

WITH RETRIEVED CONTEXT:
Based on the provided context, **semantic search** is a search technique that retrieves information based on the meaning of a query rather than relying on exact keyword matching.


## Grounding checklist

For every test, ask:

- Is the answer supported by the retrieved context?
- Are the retrieved chunks actually relevant?
- Did the model add information that was not in the context?
- Did it correctly refuse when the documents did not contain the answer?

### Main lesson

RAG can improve access to specific information, but retrieval does not automatically guarantee a perfect answer. Both retrieval and generation need testing.
